# Exploratory Data Analysis: Kepler KOI Catalog & Light Curves

This notebook explores the labeled KOI catalog and a sample of light curves
**before** any modeling decisions are made — checking class balance, missing
ephemeris data, transit depth/duration distributions, and a few example
light curves (raw, detrended, and phase-folded) to sanity-check the
preprocessing pipeline visually.

Run `python -m src.data.download_koi_catalog` and
`python -m src.data.download_light_curves --limit 50` first to have data
to explore here.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.utils.config import load_config
from src.data.preprocess import TransitParams, flatten_light_curve, phase_fold, generate_global_view, generate_local_view

cfg = load_config("../config.yaml")
koi_df = pd.read_csv(f"../{cfg.data.external_dir}/koi_cumulative.csv")
koi_df.shape

## 1. Class balance and missing ephemeris data

In [ ]:
print(koi_df["koi_disposition"].value_counts())
print("\nMissing ephemeris fields (cannot phase-fold without these):")
print(koi_df[["koi_period", "koi_time0bk", "koi_duration"]].isna().sum())

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
koi_df["koi_disposition"].value_counts().plot(kind="bar", ax=ax)
ax.set_title("KOI Disposition Class Balance")
ax.set_ylabel("count")
plt.tight_layout()
plt.show()

## 2. Transit parameter distributions (period, depth, duration)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, logscale in zip(axes, ["koi_period", "koi_depth", "koi_duration"], [True, True, False]):
    data = koi_df[col].dropna()
    ax.hist(np.log10(data) if logscale else data, bins=50)
    ax.set_title(f"{col}{' (log10)' if logscale else ''}")
plt.tight_layout()
plt.show()

## 3. Example light curve: raw -> detrended -> phase-folded -> global/local views

In [ ]:
# Pick a confirmed planet with a downloaded light curve as a worked example
confirmed = koi_df[koi_df["koi_disposition"] == "CONFIRMED"].dropna(
    subset=["koi_period", "koi_time0bk", "koi_duration"]
)
example_row = confirmed.iloc[0]
lc_path = f"../{cfg.data.raw_dir}/light_curves/kic_{int(example_row['kepid'])}.csv"
lc_df = pd.read_csv(lc_path)
time, flux = lc_df["time"].to_numpy(), lc_df["flux"].to_numpy()

params = TransitParams(example_row["koi_period"], example_row["koi_time0bk"], example_row["koi_duration"])
flat = flatten_light_curve(time, flux, method=cfg.data.detrending_method)
phase, folded = phase_fold(time, flat, params)
gview = generate_global_view(phase, folded, params.period_days, n_bins=cfg.data.global_view_bins)
lview = generate_local_view(phase, folded, params.duration_hours / 24, n_bins=cfg.data.local_view_bins,
                             num_durations=cfg.data.local_view_num_durations)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(time, flux, ".", markersize=1); axes[0, 0].set_title("Raw light curve")
axes[0, 1].plot(time, flat, ".", markersize=1); axes[0, 1].set_title("Detrended (flattened)")
axes[1, 0].plot(gview); axes[1, 0].set_title("Global view")
axes[1, 1].plot(lview); axes[1, 1].set_title("Local view (should show a dip)")
plt.tight_layout()
plt.show()

## Next steps

Once this looks sane for a handful of examples, run the full pipeline:

```bash
python -m src.data.preprocess
python -m src.training.train --smoke-test   # quick CPU sanity check
python -m src.training.train                 # full run on Colab T4
python -m src.evaluation.evaluate
```